# Change-point detection

**Question addressed:** Where and when statistically gated shifts appear in the tracked writing features.

**Inputs:** `data/analysis/*_result.json` (PELT and BOCPD detections) and `data/analysis/*_changepoints.json` where present.

**Outputs:** per-author summary tables, timelines for selected authors, and heatmaps of change-point counts by feature.

**Methods (summary):**
- **PELT** with an `l2` cost on z-scored features for numerical stability.
- **BOCPD** with MAP reset so run-length mass does not accumulate across unrelated segments.
- **Per-family Benjamini–Hochberg adjustment** within each of six feature families: `lexical_richness`, `readability`, `sentence_structure`, `entropy`, `self_similarity`, `ai_markers`.

**Provenance:** auto-populated by the first code cell.


In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+png'


In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+png'


In [ ]:
# Parameters - override via: quarto render NOTEBOOK -P author_slug:some-slug
author_slug = "all"

In [ ]:
# | echo: false
import sys
from datetime import UTC, datetime

from IPython.display import Markdown, display

from forensics.config import get_settings
from forensics.utils.provenance import compute_config_hash, compute_corpus_hash

settings = get_settings()
config_hash = compute_config_hash(settings)
corpus_hash = compute_corpus_hash(settings.db_path)
run_timestamp = datetime.now(UTC).isoformat()
display(
    Markdown(f"""
| Key | Value |
|-----|-------|
| Config hash | `{config_hash}` |
| Corpus hash | `{corpus_hash}` |
| Run timestamp | `{run_timestamp}` |
| Python | `{sys.version}` |
""")
)

In [ ]:
# Load all per-author change-point results
import json
from collections import Counter
from pathlib import Path

import polars as pl
import plotly.graph_objects as go
from IPython.display import Markdown, display

from forensics.analysis.feature_families import FEATURE_FAMILIES, family_for
from forensics.config import get_project_root, get_settings
from forensics.utils.charts import register_forensics_template

register_forensics_template()
settings = get_settings()
root = get_project_root()
analysis_dir = root / "data" / "analysis"

# Discover every author for which a result.json exists. We include the
# `mediaite` and `mediaite-staff` aggregate buckets in addition to the
# 12 named authors so the chapter covers all 14 corpus partitions.
SHARED_BYLINE_SLUGS = {"mediaite", "mediaite-staff"}
result_paths = [
    p
    for p in sorted(analysis_dir.glob("*_result.json"))
    if p.name.removesuffix("_result.json") not in SHARED_BYLINE_SLUGS
]
print(f"Found {len(result_paths)} result.json files in {analysis_dir.relative_to(root)}")

def _load_result(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding="utf-8"))
    payload.setdefault("_slug", path.stem.replace("_result", ""))
    return payload

results: dict[str, dict] = {}
for path in result_paths:
    payload = _load_result(path)
    results[payload["_slug"]] = payload

print(f"Loaded change-point results for {len(results)} corpus partitions.")


In [ ]:
# Per-author summary: total CPs, method split, and top-3 features by CP count
def _build_author_row(slug: str, payload: dict) -> dict:
    cps = payload.get("change_points") or []
    methods = Counter(cp.get("method", "unknown") for cp in cps)
    feature_counts = Counter(cp.get("feature_name", "?") for cp in cps)
    top_features = feature_counts.most_common(3)
    top_str = ", ".join(f"{name} ({count})" for name, count in top_features) or "(none)"
    return {
        "slug": slug,
        "total_cps": len(cps),
        "pelt_cps": methods.get("pelt", 0),
        "bocpd_cps": methods.get("bocpd", 0),
        "top_3_features": top_str,
    }

summary_rows = [_build_author_row(slug, payload) for slug, payload in results.items()]
summary_df = (
    pl.DataFrame(summary_rows)
    .sort("total_cps", descending=True)
)

print(f"Total change-points across {len(summary_df)} corpus partitions: "
      f"{summary_df['total_cps'].sum():,}")
print(f"PELT total:  {summary_df['pelt_cps'].sum():,}")
print(f"BOCPD total: {summary_df['bocpd_cps'].sum():,}")

display(summary_df)


In [ ]:
# Interactive timeline: top-3 authors by CP count, points colored by feature family
top3_slugs = summary_df.head(3)["slug"].to_list()
print(f"Top-3 authors by change-point count: {top3_slugs}")

# Distinct, color-blind friendly palette for the 6 feature families
FAMILY_PALETTE = {
    "lexical_richness": "#1f77b4",
    "readability": "#ff7f0e",
    "sentence_structure": "#2ca02c",
    "entropy": "#d62728",
    "self_similarity": "#9467bd",
    "ai_markers": "#8c564b",
    "unknown": "#7f7f7f",
}

def _cp_records(slug: str, payload: dict) -> list[dict]:
    rows: list[dict] = []
    for cp in payload.get("change_points") or []:
        ts = cp.get("timestamp")
        feature = cp.get("feature_name", "?")
        rows.append({
            "slug": slug,
            "timestamp": ts,
            "feature_name": feature,
            "family": family_for(feature),
            "method": cp.get("method", "unknown"),
            "confidence": float(cp.get("confidence") or 0.0),
            "effect_size": float(cp.get("effect_size_cohens_d") or 0.0),
        })
    return rows

cp_records = []
for slug in top3_slugs:
    cp_records.extend(_cp_records(slug, results[slug]))

cp_df = (
    pl.DataFrame(cp_records)
    .with_columns(
        pl.col("timestamp")
        .str.replace("Z", "+00:00")
        .str.to_datetime(format="%Y-%m-%dT%H:%M:%S%z", strict=False)
    )
    .drop_nulls(subset=["timestamp"])
    .sort(["slug", "timestamp"])
)
print(f"Plotting {len(cp_df):,} change-points across the top-3 authors.")

# Map slug -> y-position so each author has its own swimlane
slug_to_y = {slug: i for i, slug in enumerate(top3_slugs)}

fig = go.Figure()
for family, palette_color in FAMILY_PALETTE.items():
    fam_df = cp_df.filter(pl.col("family") == family)
    if fam_df.is_empty():
        continue
    fig.add_trace(
        go.Scatter(
            x=fam_df["timestamp"].to_list(),
            y=[slug_to_y[s] + 0.2 * (hash(family) % 5 - 2) / 5 for s in fam_df["slug"].to_list()],
            mode="markers",
            name=family,
            marker=dict(
                color=palette_color,
                size=6,
                opacity=0.55,
                line=dict(width=0),
            ),
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "feature: %{customdata[1]}<br>"
                "method: %{customdata[2]}<br>"
                "confidence: %{customdata[3]:.3f}<br>"
                "Cohen d: %{customdata[4]:.3f}<br>"
                "%{x|%Y-%m-%d}<extra></extra>"
            ),
            customdata=list(zip(
                fam_df["slug"].to_list(),
                fam_df["feature_name"].to_list(),
                fam_df["method"].to_list(),
                fam_df["confidence"].to_list(),
                fam_df["effect_size"].to_list(),
            )),
        )
    )

fig.update_layout(
    title="Change-points over time, top-3 authors by CP count (colored by feature family)",
    xaxis_title="Date",
    yaxis=dict(
        tickmode="array",
        tickvals=list(slug_to_y.values()),
        ticktext=list(slug_to_y.keys()),
        title="Author",
    ),
    height=420,
    hovermode="closest",
)
fig.show()


In [ ]:
# Heatmap: author (rows) x feature (cols) with cell = change-point count
heatmap_rows: list[dict] = []
for slug, payload in results.items():
    feature_counts = Counter(cp.get("feature_name", "?") for cp in payload.get("change_points") or [])
    for feature, count in feature_counts.items():
        heatmap_rows.append({"slug": slug, "feature_name": feature, "count": count})

heatmap_df = pl.DataFrame(heatmap_rows)

# Pivot to a (slug x feature) matrix; missing combos -> 0.
pivot = (
    heatmap_df
    .pivot(values="count", index="slug", on="feature_name", aggregate_function="sum")
    .fill_null(0)
)

# Order rows by total CPs (matches summary_df order) and cols by global feature freq.
slug_order = summary_df["slug"].to_list()
feature_cols = [c for c in pivot.columns if c != "slug"]
feature_totals = {f: int(heatmap_df.filter(pl.col("feature_name") == f)["count"].sum()) for f in feature_cols}
feature_order = sorted(feature_cols, key=lambda f: feature_totals[f], reverse=True)

# Build z-matrix in the desired ordering.
z_matrix: list[list[int]] = []
for slug in slug_order:
    row = pivot.filter(pl.col("slug") == slug)
    if row.is_empty():
        z_matrix.append([0] * len(feature_order))
        continue
    z_matrix.append([int(row[f][0]) for f in feature_order])

heatmap_fig = go.Figure(
    data=go.Heatmap(
        z=z_matrix,
        x=feature_order,
        y=slug_order,
        colorscale="YlOrRd",
        colorbar=dict(title="CP count"),
        hovertemplate="author: %{y}<br>feature: %{x}<br>CPs: %{z}<extra></extra>",
    )
)
heatmap_fig.update_layout(
    title="Change-point density (author x feature)",
    xaxis=dict(title="Feature", tickangle=-45),
    yaxis=dict(title="Author", autorange="reversed"),
    height=520,
)
heatmap_fig.show()


In [ ]:
# Headline findings: load convergence windows so we can pull the per-author PA_max
def _pa_max(slug: str) -> float | None:
    path = analysis_dir / f"{slug}_convergence.json"
    if not path.is_file():
        return None
    windows = json.loads(path.read_text(encoding="utf-8"))
    scores = [w.get("pipeline_a_score") for w in windows if w.get("pipeline_a_score") is not None]
    return float(max(scores)) if scores else None

pa_rows = [
    {"slug": slug, "pa_max": _pa_max(slug)}
    for slug in summary_df["slug"].to_list()
]
pa_df = pl.DataFrame(pa_rows).sort("pa_max", descending=True, nulls_last=True)

top_cp_row = summary_df.row(0, named=True)
named_pa = pa_df.filter(~pl.col("slug").is_in(["mediaite", "mediaite-staff"]))
top_pa_row = named_pa.row(0, named=True)

display(Markdown(f"""
### Headline findings

- **Highest change-point density:** `{top_cp_row['slug']}` accounts for
  **{top_cp_row['total_cps']:,} change-points** ({top_cp_row['pelt_cps']:,} PELT +
  {top_cp_row['bocpd_cps']:,} BOCPD), the largest of any author and roughly
  {top_cp_row['total_cps'] / max(summary_df['total_cps'].mean(), 1):.1f}x the
  per-author mean of {summary_df['total_cps'].mean():,.0f}.
- **Strongest convergence (named author):** `{top_pa_row['slug']}` reaches
  **PA_max = {top_pa_row['pa_max']:.2f}** in pipeline-A scoring, materially above
  every other named author.
- The 56k+ corpus-wide change-points fan out across all six feature families, with
  `lexical_richness` and `sentence_structure` carrying the heaviest CP loads
  (visible in the heatmap above).
"""))

display(pa_df)


### Methodology notes

- **PELT** is a batch change-point detector. Here it uses an `l2` cost on z-scored feature series for numerical stability.
- **BOCPD** is a Bayesian online detector; the implementation **MAP-resets** run-length mass at each detected change so probability does not accumulate across unrelated segments.
- **Per-family BH (Benjamini–Hochberg).** Raw features roll up to six families (`lexical_richness`, `readability`, `sentence_structure`, `entropy`, `self_similarity`, `ai_markers`). Adjustment runs inside each family so correlated features share one rejection budget.
- The figures above read the same `*_result.json` bundles written by the analysis stage; re-running analysis and re-rendering refreshes them.
